# Live Demo: Gender Wage Gap — ML Analysis

The Random Forest model was trained using **PySpark MLlib** in `analysis/ml_rf_antara.py`.

To keep the live demo fast, this notebook loads the saved PySpark outputs instead of retraining the model.


In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path
import sys

root = Path.cwd()
while not (root / "_quarto.yml").exists() and root != root.parent:
    root = root.parent

sys.path.insert(0, str(root / "analysis"))

from utils import PROCESSED, apply_theme

print(f"Repo root: {root}")
print(f"Processed folder: {PROCESSED}")

## Load PySpark Random Forest outputs

In [ ]:
rf_summary = pd.read_csv(PROCESSED / "rf_model_summary_antara.csv")
rf_imp = pd.read_csv(PROCESSED / "rf_feature_importance_antara.csv")
pdp = pd.read_csv(PROCESSED / "rf_partial_dependence_age_gender_antara.csv")

display(rf_summary)
rf_imp.head(10)

## Feature importance

In [ ]:
top15 = rf_imp.head(15).copy()

def clean_feature_label(f):
    if f == "AGE":
        return "Age"
    if f.startswith("SEX_LABEL_"):
        return f.replace("SEX_LABEL_", "")
    if f.startswith("STATE_NAME_"):
        return f.replace("STATE_NAME_", "")
    if f.startswith("RACE_LABEL_"):
        return f.replace("RACE_LABEL_", "")
    if f.startswith("OCC_GROUP_"):
        return f.replace("OCC_GROUP_", "Occupation group ")
    if f.startswith("IND_GROUP_"):
        return f.replace("IND_GROUP_", "Industry group ")
    return f

top15["label"] = top15["feature"].apply(clean_feature_label)

fig = px.bar(
    top15.sort_values("importance"),
    x="importance",
    y="label",
    orientation="h",
    color="is_gender",
    title="PySpark Random Forest Feature Importance",
    labels={"importance": "Importance", "label": "", "is_gender": "Gender feature"}
)

apply_theme(fig)
fig.show()

## Gender rank

In [ ]:
gender_rows = rf_imp[rf_imp["is_gender"] == True].copy()

if len(gender_rows) > 0:
    first_gender_rank = gender_rows.index[0] + 1
    total_features = len(rf_imp)
    print(f"Gender first appears at rank #{first_gender_rank} out of {total_features} features.")
    display(gender_rows.head())
else:
    print("No gender feature found.")

## Predicted wage by age and gender

In [ ]:
fig = px.line(
    pdp,
    x="AGE",
    y="predicted_wage",
    color="SEX",
    title="PySpark Random Forest: Predicted Wage by Age and Gender",
    labels={
        "AGE": "Age",
        "predicted_wage": "Predicted Annual Wage ($)",
        "SEX": "Gender"
    }
)

apply_theme(fig)
fig.show()

## Key takeaway

The PySpark Random Forest confirms that gender remains a strong predictor of wage after accounting for age, race, state, occupation, and industry.
